<div dir="rtl">

# 1. بارگذاری و متادیتا ها

In [1]:
#imports
from qdrant_client import QdrantClient, models
import json

from uuid import uuid5, NAMESPACE_URL

import frontmatter
import pandas as pd

from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import FastEmbedSparse
from langchain_qdrant import QdrantVectorStore, RetrievalMode

from pathlib import Path
import hashlib

from collections import Counter


In [2]:
ROOT = Path.cwd().parent
data_path = ROOT / "data"/ "kb"
posts = []
for path in data_path.glob("*.md"):
    post = frontmatter.load(path)
    if (
            post.metadata.get("doc_id",False)
            and
            post.metadata.get("access_level",False)
            and
            post.metadata["access_level"] in ["public","customer","internal"]
    ):
        posts.append(post)

    else:
        raise Exception(f"access_level or doc_id are missed")


data_frame = pd.DataFrame([x.metadata for x in posts],index= [f"document-{i}"for i in range(1,len(posts)+1)])
df = data_frame[["doc_id","is_latest","access_level"]]
df["content_length"] = [len(p.content) for p in posts]
display(df)

,doc_id,is_latest,access_level,content_length
document-1,account-security,True,public,791
document-2,api-rate-limits-v1,False,public,457
document-3,api-rate-limits-v2,True,public,617
document-4,billing-invoices,True,customer,883
document-5,community-post-backups,True,public,839
document-6,errors-auth,True,public,781
document-7,errors-storage,True,public,683
document-8,faq-general,True,public,933
document-9,getting-started,True,public,1009
document-10,oncall-runbook,True,internal,614


<div dir = "rtl">

# 2. chunking بر اساس عنوان ها


In [3]:
headers = [
    ("#","title"),
    ("##","section"),
]

header_splitter = MarkdownHeaderTextSplitter(headers_to_split_on = headers)

sections = []
for post in posts:
    post_sections =header_splitter.split_text(post.content)
    for section in post_sections:
        section.metadata.update(post.metadata)

    sections.extend(post_sections)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=0)

chunks = text_splitter.split_documents(sections)

print(f"len posts: {len(posts)}\nlen sections: {len(sections)}\nlen chunks: {len(chunks)}")


chunk_counts = {}

for chunk in chunks:
    doc_id = chunk.metadata["doc_id"]

    chunk_number = chunk_counts.get(doc_id, 0)

    chunk.metadata["chunk_id"] = f"{doc_id}-c{chunk_number:03d}"

    chunk.metadata["content_hash"] = hashlib.sha256(
        chunk.page_content.encode("utf-8")
    ).hexdigest()

    chunk_counts[doc_id] = chunk_number + 1


print("------------------\nshow few samples\n------------------")
for chunk in chunks[:2]:

    print(chunk.metadata)
    print(chunk.page_content)
    print("-" * 30)

len posts: 15
len sections: 51
len chunks: 51
------------------
show few samples
------------------
{'title': 'Account Security', 'section': 'Sign-in protection', 'doc_id': 'account-security', 'product_area': 'auth', 'version': 1, 'is_latest': True, 'access_level': 'public', 'updated_at': datetime.date(2026, 4, 2), 'chunk_id': 'account-security-c000', 'content_hash': '80b2de6156405c5d8e8814627c8314fb4912fa671738babc590fc2eb4d5918d0'}
You can add an extra check to your sign-in. After you enter your password, AbanCloud asks for a six-digit code generated by an authenticator app on your phone. To turn it on, open Settings > Security > Sign-in protection, scan the QR code with your authenticator app, and enter the code it shows. Save the recovery codes in a safe place in case you lose your phone.
------------------------------
{'title': 'Account Security', 'section': 'API key rotation', 'doc_id': 'account-security', 'product_area': 'auth', 'version': 1, 'is_latest': True, 'access_level': 

In [4]:

ids = [chunk.metadata["doc_id"] for chunk in chunks]
counter = Counter(ids)

for i,c in counter.items():
    print(f"id = {i}, sum of chunks = {c}")

id = account-security, sum of chunks = 3
id = api-rate-limits-v1, sum of chunks = 3
id = api-rate-limits-v2, sum of chunks = 3
id = billing-invoices, sum of chunks = 4
id = community-post-backups, sum of chunks = 4
id = errors-auth, sum of chunks = 4
id = errors-storage, sum of chunks = 3
id = faq-general, sum of chunks = 4
id = getting-started, sum of chunks = 4
id = oncall-runbook, sum of chunks = 3
id = plans-and-pricing, sum of chunks = 3
id = sales-discount-policy, sum of chunks = 3
id = sdk-configuration, sum of chunks = 4
id = storage-backups, sum of chunks = 3
id = storage-quotas, sum of chunks = 3


<div dir="rtl">

در ابتدا با تقسیم بر اساس عنوان مطمئن شدیم که محتوا بعد از برش خوردن به روش بازگشتی قاطی نمی شود یعنی نصف یک ارور در چانک دیگر نمی آید ، چرا که هر سکشن را از قبل جدا کرده بودیم
بعد با روش بازگشتی توانستیم طول چانک ها رو کنترل کنیم و الان مطمئن هستیم هیچ چانکی به موضوعات بی ربط آلوده نشده

# 3. Qdrant

In [5]:
dense_embeddings = HuggingFaceEmbeddings(
    model_name= "BAAI/bge-small-en-v1.5",
    encode_kwargs = {"normalize_embeddings": True}
)

sparse_embeddings = FastEmbedSparse(
    model_name= "Qdrant/bm25"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [6]:
dense_dimension = len(
    dense_embeddings.embed_query("test")
)

print("Dense dimension:", dense_dimension)


client = QdrantClient(path=".week20_qdrant")
collection_name = "support_rag"


if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,

        vectors_config={
            "dense": models.VectorParams(
                size=dense_dimension,
                distance=models.Distance.COSINE
            )
        },

        sparse_vectors_config={
            "sparse": models.SparseVectorParams(
                modifier=models.Modifier.IDF
            )
        }
    )

Dense dimension: 384


In [7]:

vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,

    embedding=dense_embeddings,
    sparse_embedding=sparse_embeddings,

    vector_name="dense",
    sparse_vector_name="sparse",

    retrieval_mode=RetrievalMode.HYBRID
)

In [8]:
point_ids = [
    str(uuid5(NAMESPACE_URL, chunk.metadata["chunk_id"]))
    for chunk in chunks
]

print(point_ids[:3])

vector_store.add_documents(chunks, ids=point_ids)

first_count = client.count(
    collection_name=collection_name,
    exact=True
).count

print("Count after first indexing:", first_count)

['fbb32362-3613-5783-9161-5d3acf9b6ab2', 'a307a085-c12a-561d-a70c-8695cdc69634', '2b8d9f35-929d-5213-bd9f-b6701e88c14d']
Count after first indexing: 51


In [9]:
vector_store.add_documents(chunks, ids=point_ids)

second_count = client.count(
    collection_name=collection_name,
    exact=True
).count

print("Count after second indexing:", second_count)

assert first_count == second_count
print("Idempotency check passed")

Count after second indexing: 51
Idempotency check passed


# 4. Role access

In [10]:

ROLE_ACCESS = {
    "anonymous": ["public"],
    "customer": ["public", "customer"],
    "staff": ["public", "customer", "internal"],
}


def build_filter(role: str, latest_only: bool = True) -> models.Filter:
    if role not in ROLE_ACCESS:
        raise ValueError(f"Unknown role: {role}")

    allowed_levels = ROLE_ACCESS[role]

    conditions = [
        models.FieldCondition(
            key="metadata.access_level",
            match=models.MatchAny(any=allowed_levels)
        )
    ]

    if latest_only:
        conditions.append(
            models.FieldCondition(
                key="metadata.is_latest",
                match=models.MatchValue(value=True)
            )
        )

    return models.Filter(must=conditions)


In [11]:
customer_filter = build_filter("customer")
print(customer_filter)


should=None min_should=None must=[FieldCondition(key='metadata.access_level', match=MatchAny(any=['public', 'customer']), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None), FieldCondition(key='metadata.is_latest', match=MatchValue(value=True), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None)] must_not=None


<div dir="rtl">

# تست تله های مربوط به نقش q11&q09

In [15]:
questions_path = ROOT / "eval" / "questions.jsonl"

with open(questions_path, encoding="utf-8") as file:
    questions = [json.loads(line) for line in file if line.strip()]

questions_by_id = {question["id"]: question for question in questions}

print(questions_by_id["q09"]["question"])


What is the internal phone extension for the on-call infrastructure lead?


In [13]:
for question_id in ["q09", "q10"]:
    question = questions_by_id[question_id]

    results = vector_store.similarity_search(
        query=question["question"],
        k=10,
        filter=build_filter(question["role"])
    )

    print(f"\n{question_id} | role: {question['role']}")

    for chunk in results:
        print(
            chunk.metadata["doc_id"],
            "|",
            chunk.metadata["access_level"]
        )

    assert all(
        chunk.metadata["access_level"] != "internal"
        for chunk in results
    ), f"Internal document leaked for {question_id}"

    print("Leak test passed")



q09 | role: customer
account-security | public
sdk-configuration | public
errors-storage | public
getting-started | public
faq-general | public
api-rate-limits-v2 | public
plans-and-pricing | public
billing-invoices | customer
community-post-backups | public
faq-general | public
Leak test passed

q10 | role: customer
billing-invoices | customer
storage-quotas | customer
plans-and-pricing | public
storage-backups | public
plans-and-pricing | public
sdk-configuration | public
api-rate-limits-v2 | public
faq-general | public
faq-general | public
errors-storage | public
Leak test passed


In [14]:
question = questions_by_id["q11"]

results = vector_store.similarity_search(
    query=question["question"],
    k=10,
    filter=build_filter(question["role"])
)

print(f"\nq11 | role: {question['role']}")

for chunk in results:
    print(
        chunk.metadata["doc_id"],
        "|",
        chunk.metadata.get("section"),
        "|",
        chunk.metadata["access_level"]
    )

found_correct_chunk = any(
    chunk.metadata["doc_id"] == "oncall-runbook"
    and chunk.metadata.get("section") == "Escalation contacts"
    for chunk in results
)

assert found_correct_chunk, "Expected staff document was not retrieved"
print("Staff access test passed")



q11 | role: staff
oncall-runbook | Escalation contacts | internal
account-security | Sign-in protection | public
sdk-configuration | Regions | public
errors-storage | ST-5040 | public
sales-discount-policy | Confidentiality | internal
getting-started | Creating your first project | public
faq-general | Where can I get help? | public
api-rate-limits-v2 | Default limits | public
plans-and-pricing | What each plan includes | public
billing-invoices | Invoice schedule | customer
Staff access test passed
